# STEP 31A v2 - IEEE Final Evaluation

This notebook reads the labeled file created by STEP 31B and generates the figures and tables for Section 5.2.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

ROOT=Path.cwd().resolve()
CSV=ROOT/'31B_GroundTruth_Integration'/'evaluation_summary_with_labels.csv'
OUT=ROOT/'31A_IEEE_Final_Figures'
OUT.mkdir(exist_ok=True)
df=pd.read_csv(CSV,encoding='utf-8-sig')
df=df[df['ground_truth'].isin(['CN','AD'])].copy()
df['y_true']=(df['ground_truth']=='AD').astype(int)
df['y_prob']=pd.to_numeric(df['probability_positive'],errors='coerce')
df=df[df['y_prob'].between(0,1)]
df['y_pred']=(df['y_prob']>=0.32).astype(int)
print('Evaluated subjects:',len(df))


In [ ]:
cm=confusion_matrix(df.y_true,df.y_pred)
tn,fp,fn,tp=cm.ravel()
metrics={
'Accuracy':accuracy_score(df.y_true,df.y_pred),
'Precision':precision_score(df.y_true,df.y_pred,zero_division=0),
'Sensitivity':recall_score(df.y_true,df.y_pred,zero_division=0),
'Specificity':tn/(tn+fp) if (tn+fp)>0 else np.nan,
'F1':f1_score(df.y_true,df.y_pred,zero_division=0),
'ROC_AUC':roc_auc_score(df.y_true,df.y_prob)
}
pd.DataFrame(metrics,index=[0]).to_csv(OUT/'Table_IV_subject_level_metrics.csv',index=False)
print(pd.DataFrame(metrics,index=[0]))


In [ ]:
fig,ax=plt.subplots(figsize=(5,4))
ax.imshow(cm,cmap='Greys')
for i in range(2):
  for j in range(2):
    ax.text(j,i,str(cm[i,j]),ha='center',va='center')
ax.set_xticks([0,1]);ax.set_xticklabels(['CN','AD'])
ax.set_yticks([0,1]);ax.set_yticklabels(['CN','AD'])
ax.set_xlabel('Predicted');ax.set_ylabel('Ground Truth')
plt.tight_layout();plt.savefig(OUT/'Fig17_ConfusionMatrix.png',dpi=600);plt.show()


In [ ]:
fpr,tpr,_=roc_curve(df.y_true,df.y_prob)
auc=roc_auc_score(df.y_true,df.y_prob)
plt.figure(figsize=(5,4))
plt.plot(fpr,tpr,label=f'AUC={auc:.3f}')
plt.plot([0,1],[0,1],'--')
plt.legend();plt.xlabel('False Positive Rate');plt.ylabel('True Positive Rate')
plt.tight_layout();plt.savefig(OUT/'Fig18_ROC.png',dpi=600);plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df.loc[df.y_true==0,'y_prob'],bins=20,alpha=.6,label='CN')
plt.hist(df.loc[df.y_true==1,'y_prob'],bins=20,alpha=.6,label='AD')
plt.axvline(0.32,ls='--')
plt.legend();plt.xlabel('Predicted AD Probability')
plt.tight_layout();plt.savefig(OUT/'Fig19_ProbabilityDistribution.png',dpi=600);plt.show()
print('Finished. Files saved to',OUT)
